# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 1: Fundamentos para Sistemas Escalables de IA

### Objetivo:
Aprender los conceptos fundamentales de la escalabilidad y sostenibilidad en sistemas basados en agentes inteligentes de IA, identificando los desafíos que surgen cuando la demanda y el uso del sistema crecen (degradación de rendimiento, agotamiento de recursos, explosión de costos y mantenimiento).

### Desafíos del Crecimiento de Sistemas IA
1. **Performance degradation (Degradación de rendimiento):** El sistema responde con lentitud a medida que aumenta el número de usuarios concurrentes.
2. **Resource exhaustion (Agotamiento de recursos):** El servidor local se queda sin CPU o memoria, o bien las APIs externas (como el LLM) bloquean las consultas por límites de cuota (rate limits).
3. **Cost explosion (Explosión de costos):** Incremento drástico en la factura debido a llamadas redundantes o ineficientes al LLM.
4. **Maintenance overhead (Sobrecarga de mantenimiento):** Complejidad creciente en el mantenimiento de bases de datos de historial y configuraciones de agentes.

### Principios de un Sistema Escalable
- **Logging & Monitoring:** Registrar tiempos de respuesta y recursos consumidos de manera continua.
- **Microservicios:** Separar tareas complejas (ej. scraping de Wikipedia, parsing del LLM) en componentes independientes.
- **Message queues (Colas de mensajes):** Desacoplar tareas bloqueantes ejecutándolas asíncronamente mediante colas (ej. Celery, RabbitMQ).
- **Automation (Automatización):** pipelines automáticos de CI/CD para despliegue sin intervención humana.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Clase ScalableAgent (Agente Escalable Básico)
Implementaremos un agente simple que simula el procesamiento escalable de solicitudes de usuarios, midiendo la latencia de ejecución y registrando la actividad a través de la librería estándar `logging` de Python.


In [ ]:
import logging
import time

# Configuración básica de logs en la consola
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)

class ScalableAgent:
    def __init__(self, executor):
        self.executor = executor

    def process(self, data):
        logging.info(f"Iniciando procesamiento para la consulta: '{data}'")
        start_time = time.time()
        
        try:
            if llm is None:
                # Flujo alternativo simulado en caso de ausencia de API keys
                time.sleep(0.3)
                result = f"Respuesta simulada para: {data}"
            else:
                # Invocación real del agente de Wikipedia
                response = self.executor.invoke({"input": data})
                result = response.get("output", "")
        except Exception as e:
            result = f"Error en procesamiento: {e}"
            logging.error(f"Fallo en procesamiento de '{data}': {e}")
            
        latency = round(time.time() - start_time, 4)
        logging.info(f"Procesamiento finalizado. Latencia: {latency}s")
        return result

# Instanciar el agente escalable
scalable_agent = ScalableAgent(agent_executor)

# Probar ejecución secuencial
print("Respuesta:", scalable_agent.process("¿Quién escribió Don Quijote de la Mancha?"))


### Preguntas de Análisis
1. **¿Qué problemas de rendimiento (latencia) y costos podrían surgir si 100 usuarios invocaran el método `process` de forma simultánea en esta arquitectura secuencial?**
2. **¿De qué manera la degradación de rendimiento de una API de LLM (ej. un incremento en la latencia de respuesta de OpenAI) afecta la escalabilidad general de nuestra aplicación?**
3. **¿Cómo ayuda la arquitectura de microservicios a solucionar cuellos de botella individuales en comparación con un sistema monolítico?**
